# Carga y visualización de datos

El primero paso será importar las librerías necesarias así como importar los datos.

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/workspace/diamonds.csv')

Tras esto vamos a realizar una exploración general de los datos. 

In [2]:
print("--- Primeras 5 filas del dataset ---")
display(df.head())

print("\n--- Información general del dataset ---")
df.info()

print("\n--- Estadísticas descriptivas ---")
display(df.describe())

--- Primeras 5 filas del dataset ---


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75



--- Información general del dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53939 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53939 non-null  float64
 8   y        53940 non-null  float64
 9   z        53930 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB

--- Estadísticas descriptivas ---


,carat,depth,table,price,x,y,z
count,53940.000000,53939.000000,53940.000000,53940.000000,53939.000000,53940.000000,53930.000000
mean,0.797940,61.749413,57.457184,3932.799722,5.731153,5.734526,3.538723
std,0.474011,1.432633,2.234491,3989.439738,1.121771,1.142135,0.705762
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


# Preprocesamiento de datos
En este apartado manejaremos los valores faltantes, codificaremos las variables categóricas y estandarizaremos los valores necesarios. 

In [5]:
target_col = 'cut'
cat_features = ['color', 'clarity'] # Variables categóricas predictoras
num_features = ['carat', 'depth', 'table', 'price', 'x', 'y', 'z'] # Variables numéricas

imputer = SimpleImputer(strategy='mean')
df[num_features] = imputer.fit_transform(df[num_features])

label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

le_target = LabelEncoder()
df[target_col] = le_target.fit_transform(df[target_col])

X = df.drop(target_col, axis=1)
y = df[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Preprocesamiento completado explícitamente:")
print(f"- Columnas numéricas imputadas: {num_features}")
print(f"- Columnas categóricas codificadas: {cat_features} + target ('{target_col}')")
print("- Todas las variables predictoras (X) han sido estandarizadas.")

Preprocesamiento completado explícitamente:
- Columnas numéricas imputadas: ['carat', 'depth', 'table', 'price', 'x', 'y', 'z']
- Columnas categóricas codificadas: ['color', 'clarity'] + target ('cut')
- Todas las variables predictoras (X) han sido estandarizadas.


# División del dataset
Dividimos en dos subconjuntos, uno para entrenamiento y otro para pruebas. 

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} filas")
print(f"Tamaño del conjunto de pruebas: {X_test.shape[0]} filas")

Tamaño del conjunto de entrenamiento: 43152 filas
Tamaño del conjunto de pruebas: 10788 filas


# Implementación de modelos
Procedemos a implementar los siguientes modelos:
- Regresión logística
- K-Nearest Neighbors (KNN)
- Support Vector Classifier (SVC)
- Árbol de decisión
- Random Forest

In [9]:
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=2000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVC': SVC(random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

modelos_entrenados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    modelos_entrenados[nombre] = modelo
    print(f"Modelo '{nombre}' entrenado con éxito.")

Modelo 'Regresión Logística' entrenado con éxito.
Modelo 'KNN' entrenado con éxito.
Modelo 'SVC' entrenado con éxito.
Modelo 'Árbol de Decisión' entrenado con éxito.
Modelo 'Random Forest' entrenado con éxito.


# Evaluación del rendimiento
Para evaluar el rendimiento de cada modelo utilizaremos diferentes métricas como precisión, matriz de confusión y F1-score. 

In [12]:
f1_scores = {}

for nombre, modelo in modelos_entrenados.items():
    y_pred = modelo.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    
    report_dict = classification_report(y_test, y_pred, output_dict=True)
    f1_macro = report_dict['macro avg']['f1-score']
    f1_scores[nombre] = f1_macro
    
    # Imprimir resultados
    print(f"\n{'='*40}")
    print(f"Evaluación del modelo: {nombre}")
    print(f"{'='*40}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-Score (Macro Avg): {f1_macro:.4f}")
    print("\nMatriz de Confusión:")
    print(conf_matrix)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Evaluación del modelo: Regresión Logística
Accuracy: 0.6596
F1-Score (Macro Avg): 0.5576

Matriz de Confusión:
[[ 176   28   37   73   21]
 [  40  137  256  268  303]
 [   1    2 3765  276  248]
 [   5    3  304 2019  444]
 [  10   52  879  422 1019]]

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.53      0.62       335
           1       0.62      0.14      0.22      1004
           2       0.72      0.88      0.79      4292
           3       0.66      0.73      0.69      2775
           4       0.50      0.43      0.46      2382

    accuracy                           0.66     10788
   macro avg       0.65      0.54      0.56     10788
weighted avg       0.65      0.66      0.63     10788


Evaluación del modelo: KNN
Accuracy: 0.6655
F1-Score (Macro Avg): 0.6253

Matriz de Confusión:
[[ 255   65    4    6    5]
 [  35  512   43  136  278]
 [   1   15 3888  227  161]
 [   2   78  395 1941  359]
 [   8  292  723  776  583

# Conclusión
Procedemos a determinar cual es el mejor modelo en términos de F1 score. 

In [13]:
mejor_modelo = max(f1_scores, key=f1_scores.get)
mejor_f1 = f1_scores[mejor_modelo]

print(f"\n🏆 CONCLUSIÓN 🏆")
print(f"Tras comparar los 5 modelos de clasificación, el modelo con mejor rendimiento")
print(f"basado en la métrica F1-Score es: **{mejor_modelo}**")
print(f"Alcanzando un F1-Score de {mejor_f1:.4f}.")


🏆 CONCLUSIÓN 🏆
Tras comparar los 5 modelos de clasificación, el modelo con mejor rendimiento
basado en la métrica F1-Score es: **Random Forest**
Alcanzando un F1-Score de 0.7861.
